# Laboratorio 5 — Minería de Textos y Análisis de Sentimiento

**CC3084 — Data Science**

Notebook final completo. Incluye el flujo completo del laboratorio y los agregados solicitados:

- análisis exploratorio;
- limpieza y preprocesamiento;
- unigramas, bigramas y trigramas;
- visualizaciones y nubes de palabras;
- comparación de modelos con validación cruzada;
- selección del mejor modelo;
- función para clasificar nuevos tweets;
- análisis de sentimiento positivo, negativo y neutro;
- top 10 tweets positivos y negativos;
- comparación de negatividad entre categorías;
- variable `negativity`;
- reentrenamiento del modelo con negatividad;
- comparación final y generación de resultados para el informe.

**Importante:** ejecuta el notebook desde la carpeta raíz del proyecto, donde debe existir `data/train.csv`.


In [ ]:
# ============================================================
# LABORATORIO 5 - MINERIA DE TEXTOS Y ANALISIS DE SENTIMIENTO
# CC3084 - DATA SCIENCE
# CODIGO COMPLETO FINAL
# Ejecutar desde la carpeta raiz del proyecto, donde existe data/train.csv
# ============================================================

import os
import re
import sys
import html
import warnings
import subprocess
from collections import Counter
from functools import lru_cache
from importlib.metadata import version, PackageNotFoundError

warnings.filterwarnings("ignore")

# ------------------------------------------------------------


## 0. LIBRERIAS


In [ ]:
# ------------------------------------------------------------

def asegurar_paquete(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
    except ImportError:
        print(f"Instalando dependencia faltante: {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

for imp, pip in [
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("sklearn", "scikit-learn"),
    ("nltk", "nltk"),
    ("wordcloud", "wordcloud"),
    ("textblob", "textblob"),
    ("scipy", "scipy"),
    ("joblib", "joblib"),
]:
    asegurar_paquete(imp, pip)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import nltk

from scipy.stats import mannwhitneyu
from wordcloud import WordCloud
from textblob import TextBlob

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedGroupKFold,
    cross_validate,
    cross_val_score,
    GridSearchCV,
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)

# ------------------------------------------------------------


## 1. CONFIGURACION


In [ ]:
# ------------------------------------------------------------

RANDOM_STATE = 42
DATA_DIR = "data"
FIG_DIR = "figures"
MODEL_DIR = os.path.join(DATA_DIR, "modelos")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_KAGGLE_PATH = os.path.join(DATA_DIR, "test.csv")

if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(
        "No se encontro data/train.csv. Ejecuta este codigo desde la carpeta raiz del proyecto."
    )

plt.rcParams["figure.dpi"] = 130
sns.set_theme(style="whitegrid")

# ------------------------------------------------------------


## 2. CARGA Y ANALISIS EXPLORATORIO


In [ ]:
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("1. CARGA Y ANALISIS EXPLORATORIO")
print("=" * 80)

raw_df = pd.read_csv(TRAIN_PATH)

print("Dimensiones originales:", raw_df.shape)
print("\nTipos de datos:")
print(raw_df.dtypes)

print("\nValores nulos:")
print(raw_df.isnull().sum())

print("\nPorcentaje de nulos:")
print((raw_df.isnull().mean() * 100).round(2))

print("\nDistribucion de target:")
print(raw_df["target"].value_counts().sort_index())
print((raw_df["target"].value_counts(normalize=True).sort_index() * 100).round(2))

raw_df["text_len"] = raw_df["text"].astype(str).str.len()
raw_df["word_count"] = raw_df["text"].astype(str).str.split().str.len()

print("\nLongitud en caracteres:")
print(raw_df["text_len"].describe().round(2))

print("\nLongitud en palabras:")
print(raw_df["word_count"].describe().round(2))

print("\nDuplicados por texto:", raw_df.duplicated(subset=["text"]).sum())
print(
    "Duplicados exactos texto+target:",
    raw_df.duplicated(subset=["text", "target"]).sum(),
)

raw_df.to_csv(os.path.join(DATA_DIR, "train_explored.csv"), index=False)

# Figura 1: distribucion target
counts = raw_df["target"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["No desastre (0)", "Desastre real (1)"], counts.values)
for bar, value in zip(bars, counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 20,
        str(value),
        ha="center",
        fontweight="bold",
    )
ax.set_title("Distribucion de la variable objetivo")
ax.set_ylabel("Cantidad de tweets")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "01_distribucion_target.png"), dpi=300)
plt.show()

# ------------------------------------------------------------


## 3. LIMPIEZA Y PREPROCESAMIENTO


In [ ]:
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("2. LIMPIEZA Y PREPROCESAMIENTO")
print("=" * 80)

# Intentar usar stopwords de NLTK. Si no estan disponibles y no pueden
# descargarse, usar la lista inglesa de scikit-learn como respaldo.
try:
    from nltk.corpus import stopwords
    STOPWORDS = set(stopwords.words("english"))
    STOPWORD_SOURCE = "NLTK"
except LookupError:
    try:
        nltk.download("stopwords", quiet=True)
        from nltk.corpus import stopwords
        STOPWORDS = set(stopwords.words("english"))
        STOPWORD_SOURCE = "NLTK"
    except Exception:
        # Respaldo local equivalente a la lista inglesa habitual de NLTK.
        # Se incluye para que el script siga funcionando incluso sin Internet.
        STOPWORDS = set("""i me my myself we our ours ourselves you you're you've you'll you'd your yours yourself yourselves he him his himself she she's her hers herself it it's its itself they them their theirs themselves what which who whom this that that'll these those am is are was were be been being have has had having do does did doing a an the and but if or because as until while of at by for with about against between into through during before after above below to from up down in out on off over under again further then once here there when where why how all any both each few more most other some such no nor not only own same so than too very s t can will just don don't should should've now d ll m o re ve y ain aren aren't couldn couldn't didn didn't doesn doesn't hadn hadn't hasn hasn't haven haven't isn isn't ma mightn mightn't mustn mustn't needn needn't shan shan't shouldn shouldn't wasn wasn't weren weren't won won't wouldn wouldn't""".split())
        STOPWORD_SOURCE = "lista local equivalente a NLTK (respaldo)"

NEGATIONS = {"no", "not", "nor"}
STOPWORDS = STOPWORDS - NEGATIONS

print("Fuente de stopwords:", STOPWORD_SOURCE)

URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
HASHTAG_SYMBOL_RE = re.compile(r"#")
EMOJI_RE = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002700-\U000027BF"
    "\U0001F900-\U0001F9FF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE,
)
PUNCT_RE = re.compile(r"[^\w\s]")
NUMBER_RE = re.compile(r"\b\d+\b")
NUM_911_RE = re.compile(r"\b911\b")
MULTISPACE_RE = re.compile(r"\s+")
WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def clean_text(text: str, keep_911: bool = True) -> str:
    """Limpia el texto para el modelo de clasificacion."""
    text = html.unescape(str(text))

    # Quitar URLs/menciones antes de pasar a ASCII.
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)

    # Quitar caracteres no ASCII / mojibake y pasar a minusculas.
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.lower()

    # Mantener la palabra del hashtag, pero quitar #.
    text = HASHTAG_SYMBOL_RE.sub("", text)

    # Quitar emojis para el modelo TF-IDF.
    text = EMOJI_RE.sub(" ", text)
    text = text.replace("_", " ")

    # Proteger solamente el numero 911 como token independiente.
    if keep_911:
        text = NUM_911_RE.sub(" num911token ", text)

    # Puntuacion y numeros.
    text = PUNCT_RE.sub(" ", text)
    text = NUMBER_RE.sub(" ", text)

    if keep_911:
        text = text.replace("num911token", " 911 ")

    text = MULTISPACE_RE.sub(" ", text).strip()

    tokens = [
        token
        for token in text.split()
        if (token not in STOPWORDS) and (len(token) > 1 or token == "911")
    ]

    return " ".join(tokens)


# Eliminar solo duplicados exactos de texto + target.
df = raw_df.drop_duplicates(subset=["text", "target"]).copy().reset_index(drop=True)
print("Filas luego de quitar duplicados exactos texto+target:", len(df))

# Limpiar texto.
df["text_clean"] = df["text"].apply(clean_text)
df["clean_word_count"] = df["text_clean"].str.split().str.len()

# Eliminar textos que quedaron vacios.
empty_count = int((df["text_clean"].str.len() == 0).sum())
print("Textos vacios despues de limpieza:", empty_count)
df = df[df["text_clean"].str.len() > 0].reset_index(drop=True)

# Diagnostico de duplicados luego de limpiar.
clean_duplicate_rows = int(df.duplicated(subset=["text_clean"], keep=False).sum())
clean_duplicate_groups = int((df["text_clean"].value_counts() > 1).sum())
conflicting_clean_groups = int(
    (df.groupby("text_clean")["target"].nunique() > 1).sum()
)

print("Filas pertenecientes a textos limpios repetidos:", clean_duplicate_rows)
print("Textos limpios distintos repetidos:", clean_duplicate_groups)
print("Textos limpios con etiquetas contradictorias:", conflicting_clean_groups)

print("\nEjemplos antes/despues:")
for i in range(min(5, len(df))):
    print("\nOriginal:", df.loc[i, "text"])
    print("Limpio  :", df.loc[i, "text_clean"])

df.to_csv(os.path.join(DATA_DIR, "train_clean.csv"), index=False)

# ------------------------------------------------------------


## 4. FRECUENCIAS, UNIGRAMAS, BIGRAMAS Y TRIGRAMAS


In [ ]:
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("3. N-GRAMAS, FRECUENCIAS Y PROBABILIDADES")
print("=" * 80)


def ngram_table(corpus, ngram_range=(1, 1), top_n=20):
    vec = CountVectorizer(ngram_range=ngram_range)
    matrix = vec.fit_transform(corpus)
    counts = np.asarray(matrix.sum(axis=0)).ravel()
    terms = np.asarray(vec.get_feature_names_out())
    order = np.argsort(counts)[::-1][:top_n]
    selected_counts = counts[order]
    total = counts.sum()

    return pd.DataFrame(
        {
            "ngram": terms[order],
            "frecuencia": selected_counts,
            "probabilidad": selected_counts / total if total else 0,
        }
    )


desastre = df.loc[df["target"] == 1, "text_clean"]
no_desastre = df.loc[df["target"] == 0, "text_clean"]

uni_d = ngram_table(desastre, (1, 1), 20)
uni_nd = ngram_table(no_desastre, (1, 1), 20)
bi_d = ngram_table(desastre, (2, 2), 15)
bi_nd = ngram_table(no_desastre, (2, 2), 15)
tri_d = ngram_table(desastre, (3, 3), 10)
tri_nd = ngram_table(no_desastre, (3, 3), 10)

print("\nTOP 20 UNIGRAMAS - DESASTRE")
print(uni_d.to_string(index=False))
print("\nTOP 20 UNIGRAMAS - NO DESASTRE")
print(uni_nd.to_string(index=False))
print("\nTOP 15 BIGRAMAS - DESASTRE")
print(bi_d.to_string(index=False))
print("\nTOP 15 BIGRAMAS - NO DESASTRE")
print(bi_nd.to_string(index=False))
print("\nTOP 10 TRIGRAMAS - DESASTRE")
print(tri_d.to_string(index=False))
print("\nTOP 10 TRIGRAMAS - NO DESASTRE")
print(tri_nd.to_string(index=False))

uni_d.rename(columns={"ngram": "palabra"}).to_csv(
    os.path.join(DATA_DIR, "unigramas_desastre.csv"), index=False
)
uni_nd.rename(columns={"ngram": "palabra"}).to_csv(
    os.path.join(DATA_DIR, "unigramas_no_desastre.csv"), index=False
)
bi_d.rename(columns={"ngram": "bigrama"}).to_csv(
    os.path.join(DATA_DIR, "bigramas_desastre.csv"), index=False
)
bi_nd.rename(columns={"ngram": "bigrama"}).to_csv(
    os.path.join(DATA_DIR, "bigramas_no_desastre.csv"), index=False
)
tri_d.rename(columns={"ngram": "trigrama"}).to_csv(
    os.path.join(DATA_DIR, "trigramas_desastre.csv"), index=False
)
tri_nd.rename(columns={"ngram": "trigrama"}).to_csv(
    os.path.join(DATA_DIR, "trigramas_no_desastre.csv"), index=False
)

# Palabras comunes entre top 50 de ambas categorias.
top50_d = set(ngram_table(desastre, (1, 1), 50)["ngram"])
top50_nd = set(ngram_table(no_desastre, (1, 1), 50)["ngram"])
comunes = sorted(top50_d & top50_nd)
print("\nPalabras presentes en el top 50 de ambas categorias:")
print(comunes)

# ------------------------------------------------------------


## 5. VISUALIZACIONES EDA


In [ ]:
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("4. GENERANDO FIGURAS EDA")
print("=" * 80)

# Nubes de palabras.
for filename, title, text_series in [
    ("02_wordcloud_desastre.png", "Nube de palabras - Desastre real", desastre),
    ("03_wordcloud_no_desastre.png", "Nube de palabras - No desastre", no_desastre),
    ("04_wordcloud_general.png", "Nube de palabras - Dataset completo", df["text_clean"]),
]:
    wc = WordCloud(width=900, height=500, background_color="white", max_words=100).generate(
        " ".join(text_series)
    )
    plt.figure(figsize=(10, 6))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.show()

# Top palabras.
for table, filename, title in [
    (uni_d.head(15), "05_top_palabras_desastre.png", "Top 15 palabras - Desastre real"),
    (uni_nd.head(15), "06_top_palabras_no_desastre.png", "Top 15 palabras - No desastre"),
    (bi_d.head(10), "07_top_bigramas_desastre.png", "Top 10 bigramas - Desastre real"),
]:
    plot_df = table.iloc[::-1]
    plt.figure(figsize=(8, 6))
    plt.barh(plot_df["ngram"], plot_df["frecuencia"])
    plt.title(title)
    plt.xlabel("Frecuencia")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.show()

# Longitud de tweets.
plt.figure(figsize=(8, 5))
plt.hist(
    df.loc[df["target"] == 1, "clean_word_count"],
    bins=20,
    alpha=0.6,
    label="Desastre real",
)
plt.hist(
    df.loc[df["target"] == 0, "clean_word_count"],
    bins=20,
    alpha=0.6,
    label="No desastre",
)
plt.title("Distribucion del numero de palabras por tweet")
plt.xlabel("Numero de palabras")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "08_longitud_tweets.png"), dpi=300)
plt.show()

# Keywords.
kw = df.dropna(subset=["keyword"])
if not kw.empty:
    top_kw_d = kw.loc[kw["target"] == 1, "keyword"].value_counts().head(10)
    top_kw_nd = kw.loc[kw["target"] == 0, "keyword"].value_counts().head(10)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    top_kw_d.sort_values().plot.barh(ax=axes[0])
    axes[0].set_title("Top 10 keywords - Desastre real")
    top_kw_nd.sort_values().plot.barh(ax=axes[1])
    axes[1].set_title("Top 10 keywords - No desastre")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "09_top_keywords.png"), dpi=300)
    plt.show()

# ------------------------------------------------------------


## 6. DIVISION TRAIN/TEST SIN FUGA POR TEXTOS DUPLICADOS


In [ ]:
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("5. DIVISION TRAIN/TEST")
print("=" * 80)

# Se usa el texto limpio como grupo. Asi, dos tweets que se vuelven iguales
# despues del preprocesamiento nunca quedan uno en train y otro en test.
outer_split = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(
    outer_split.split(
        X=df["text_clean"],
        y=df["target"],
        groups=df["text_clean"],
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

X_train = train_df["text_clean"]
y_train = train_df["target"]
X_test = test_df["text_clean"]
y_test = test_df["target"]
groups_train = train_df["text_clean"]

print("Entrenamiento:", len(train_df))
print("Prueba:", len(test_df))
print("Proporcion target=1 train:", round(y_train.mean(), 4))
print("Proporcion target=1 test :", round(y_test.mean(), 4))
print(
    "Textos limpios compartidos entre train y test:",
    len(set(X_train) & set(X_test)),
)

# Guardar la particion para reproducibilidad.
train_df.to_csv(os.path.join(DATA_DIR, "train_split.csv"), index=False)
test_df.to_csv(os.path.join(DATA_DIR, "test_split.csv"), index=False)

# ------------------------------------------------------------


## 7. COMPARACION DE MODELOS CON PIPELINE + VALIDACION CRUZADA


In [ ]:
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("6. COMPARACION DE MODELOS CON VALIDACION CRUZADA")
print("=" * 80)

# El TF-IDF vive dentro del Pipeline. Esto evita leakage durante CV.
def crear_pipeline(clf):
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    max_features=5000,
                    ngram_range=(1, 2),
                    min_df=1,
                ),
            ),
            ("clf", clf),
        ]
    )


modelos = {
    "Regresion Logistica": crear_pipeline(
        LogisticRegression(max_iter=1500, random_state=RANDOM_STATE)
    ),
    "Naive Bayes Multinomial": crear_pipeline(MultinomialNB()),
    "SVM lineal (LinearSVC)": crear_pipeline(LinearSVC(random_state=RANDOM_STATE)),
    "Random Forest": crear_pipeline(
        RandomForestClassifier(
            n_estimators=200,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
}

cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE,
)

SCORING = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
}

filas_cv = []

for nombre, pipeline in modelos.items():
    res = cross_validate(
        pipeline,
        X_train,
        y_train,
        groups=groups_train,
        cv=cv,
        scoring=SCORING,
        n_jobs=1,
        return_train_score=False,
    )

    fila = {"modelo": nombre}
    for metric in SCORING:
        fila[metric] = res[f"test_{metric}"].mean()
        fila[f"{metric}_std"] = res[f"test_{metric}"].std()
    filas_cv.append(fila)

    print(f"\n{nombre}")
    print(f"  Accuracy CV      : {fila['accuracy']:.4f} +/- {fila['accuracy_std']:.4f}")
    print(f"  Precision macro  : {fila['precision_macro']:.4f} +/- {fila['precision_macro_std']:.4f}")
    print(f"  Recall macro     : {fila['recall_macro']:.4f} +/- {fila['recall_macro_std']:.4f}")
    print(f"  F1 macro         : {fila['f1_macro']:.4f} +/- {fila['f1_macro_std']:.4f}")

cv_df = pd.DataFrame(filas_cv).sort_values("f1_macro", ascending=False).reset_index(drop=True)
cv_df.to_csv(os.path.join(DATA_DIR, "resultados_cv_modelos.csv"), index=False)

print("\nResumen CV:")
print(cv_df.round(4).to_string(index=False))

# Figura comparativa CV.
plot_cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
cv_df.set_index("modelo")[plot_cols].plot.bar(figsize=(10, 6), width=0.8)
plt.title("Comparacion de modelos con validacion cruzada")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=18)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "10_cv_comparacion_modelos.png"), dpi=300)
plt.show()

# ------------------------------------------------------------


## 8. GRID SEARCH EN TRAIN: SELECCION DEL MEJOR MODELO SIN MIRAR TEST


In [ ]:
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("7. AJUSTE DE HIPERPARAMETROS Y SELECCION DEL MEJOR MODELO")
print("=" * 80)

param_grids = {
    "Regresion Logistica": {
        "clf__C": [0.1, 1.0, 10.0],
        "clf__class_weight": [None, "balanced"],
    },
    "Naive Bayes Multinomial": {
        "clf__alpha": [0.1, 0.5, 1.0],
    },
    "SVM lineal (LinearSVC)": {
        "clf__C": [0.1, 1.0, 10.0],
        "clf__class_weight": [None, "balanced"],
    },
    "Random Forest": {
        "clf__n_estimators": [200, 300],
        "clf__max_depth": [None, 30],
    },
}

# Para que el tiempo sea razonable, ajustamos los modelos lineales/bayesianos
# con mejor F1 CV. Random Forest ya queda comparado en CV, pero no entra al GridSearch.
top_model_names = [n for n in cv_df["modelo"].tolist() if n != "Random Forest"][:3]

best_searches = {}
search_rows = []

for nombre in top_model_names:
    print(f"\nGridSearch: {nombre}")
    gs = GridSearchCV(
        estimator=modelos[nombre],
        param_grid=param_grids[nombre],
        scoring="f1_macro",
        cv=cv,
        n_jobs=1,
        refit=True,
    )
    gs.fit(X_train, y_train, groups=groups_train)
    best_searches[nombre] = gs

    search_rows.append(
        {
            "modelo": nombre,
            "best_f1_macro_cv": gs.best_score_,
            "best_params": str(gs.best_params_),
        }
    )

    print("Mejores parametros:", gs.best_params_)
    print("Mejor F1 macro CV:", round(gs.best_score_, 4))

search_df = pd.DataFrame(search_rows).sort_values(
    "best_f1_macro_cv", ascending=False
).reset_index(drop=True)
search_df.to_csv(os.path.join(DATA_DIR, "resultados_gridsearch.csv"), index=False)

mejor_nombre = search_df.loc[0, "modelo"]
mejor_pipeline = best_searches[mejor_nombre].best_estimator_
mejores_parametros = best_searches[mejor_nombre].best_params_
mejor_f1_cv = best_searches[mejor_nombre].best_score_

print("\nMEJOR MODELO SEGUN CV:", mejor_nombre)
print("F1 macro CV:", round(mejor_f1_cv, 4))
print("Parametros:", mejores_parametros)

# ------------------------------------------------------------


## 9. EVALUACION FINAL DEL MEJOR MODELO EN TEST


In [ ]:
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("8. EVALUACION FINAL SOBRE EL CONJUNTO DE PRUEBA")
print("=" * 80)

# Si el ganador es SVM, calibrarlo usando SOLO train para obtener probabilidades.
if mejor_nombre == "SVM lineal (LinearSVC)":
    modelo_final_base = CalibratedClassifierCV(
        clone(mejor_pipeline),
        cv=3,
        method="sigmoid",
    )
else:
    modelo_final_base = clone(mejor_pipeline)

modelo_final_base.fit(X_train, y_train)
y_pred_base = modelo_final_base.predict(X_test)

base_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_base),
    "precision_macro": precision_score(y_test, y_pred_base, average="macro"),
    "recall_macro": recall_score(y_test, y_pred_base, average="macro"),
    "f1_macro": f1_score(y_test, y_pred_base, average="macro"),
    "precision_desastre": precision_score(y_test, y_pred_base, pos_label=1),
    "recall_desastre": recall_score(y_test, y_pred_base, pos_label=1),
    "f1_desastre": f1_score(y_test, y_pred_base, pos_label=1),
}

print("\nModelo:", mejor_nombre)
print(pd.Series(base_metrics).round(4))
print("\nReporte de clasificacion:")
print(
    classification_report(
        y_test,
        y_pred_base,
        target_names=["No desastre", "Desastre real"],
        digits=4,
    )
)

# Matriz de confusion.
cm = confusion_matrix(y_test, y_pred_base)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No desastre", "Desastre"],
    yticklabels=["No desastre", "Desastre"],
)
plt.title(f"Matriz de confusion - {mejor_nombre}")
plt.xlabel("Predicho")
plt.ylabel("Real")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "11_matriz_confusion_mejor.png"), dpi=300)
plt.show()

# ROC solo si hay probabilidades.
roc_auc = np.nan
if hasattr(modelo_final_base, "predict_proba"):
    proba_base = modelo_final_base.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, proba_base)
    fpr, tpr, _ = roc_curve(y_test, proba_base)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, lw=2, label=f"ROC (AUC = {roc_auc:.4f})")
    plt.plot([0, 1], [0, 1], "--", lw=1)
    plt.title(f"Curva ROC - {mejor_nombre}")
    plt.xlabel("Tasa de falsos positivos")
    plt.ylabel("Tasa de verdaderos positivos")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "12_roc_mejor_modelo.png"), dpi=300)
    plt.show()
    print("ROC-AUC:", round(roc_auc, 4))

# ------------------------------------------------------------


## 10. FUNCION DE CLASIFICACION BASE


In [ ]:
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("9. FUNCION DE CLASIFICACION DE NUEVOS TWEETS")
print("=" * 80)


def clasificar_tweet_base(texto: str):
    limpio = clean_text(texto)
    pred = int(modelo_final_base.predict([limpio])[0])

    if hasattr(modelo_final_base, "predict_proba"):
        probs = modelo_final_base.predict_proba([limpio])[0]
        confianza = float(probs[pred])
    else:
        confianza = np.nan

    etiqueta = "DESASTRE" if pred == 1 else "NO DESASTRE"
    return pred, etiqueta, confianza, limpio


ejemplos = [
    "Forest fire near La Ronge Sask. Canada",
    "I love this new phone, it's fire!",
    "BREAKING: 7.1 magnitude earthquake hits Mexico City",
    "That concert was the bomb, best night ever",
    "Please call 911, there is a wreck on the highway",
]

for tweet in ejemplos:
    pred, etiqueta, confianza, limpio = clasificar_tweet_base(tweet)
    print(f"\nTweet: {tweet}")
    print(f"Limpio: {limpio}")
    if np.isnan(confianza):
        print(f"Clasificacion: {etiqueta}")
    else:
        print(f"Clasificacion: {etiqueta} | confianza={confianza:.3f}")

# ------------------------------------------------------------


## 11. ANALISIS DE SENTIMIENTO CON TEXTBLOB


In [ ]:
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("10. ANALISIS DE SENTIMIENTO")
print("=" * 80)


def preparar_texto_sentimiento(text: str) -> str:
    """Limpieza ligera: conserva puntuacion, mayusculas y emoticones."""
    text = html.unescape(str(text))
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = text.replace("#", "")
    return MULTISPACE_RE.sub(" ", text).strip()


@lru_cache(maxsize=None)
def polaridad_palabra(word: str) -> float:
    return float(TextBlob(word).sentiment.polarity)


def analizar_sentimiento(text: str) -> pd.Series:
    text_sent = preparar_texto_sentimiento(text)
    sentiment = TextBlob(text_sent).sentiment
    polarity = float(sentiment.polarity)
    subjectivity = float(sentiment.subjectivity)

    if polarity > 0.05:
        label = "Positivo"
    elif polarity < -0.05:
        label = "Negativo"
    else:
        label = "Neutro"

    # Negatividad queda entre 0 y 1; 1 = muy negativo.
    negativity = float(np.clip(-polarity, 0, 1))
    positivity = float(np.clip(polarity, 0, 1))

    tokens = [
        w.lower()
        for w in WORD_RE.findall(text_sent)
        if len(w) > 1 and w.lower() not in STOPWORDS
    ]
    positivas, negativas, neutras = [], [], []

    for word in tokens:
        p = polaridad_palabra(word)
        if p > 0:
            positivas.append(word)
        elif p < 0:
            negativas.append(word)
        else:
            neutras.append(word)

    return pd.Series(
        {
            "text_sentiment": text_sent,
            "sentiment_polarity": polarity,
            "sentiment_subjectivity": subjectivity,
            "positivity": positivity,
            "negativity": negativity,
            "sentiment_label": label,
            "n_positive_words": len(positivas),
            "n_negative_words": len(negativas),
            "n_neutral_words": len(neutras),
            "positive_words": positivas,
            "negative_words": negativas,
            "neutral_words": neutras,
        }
    )


sentiment_columns = df["text"].apply(analizar_sentimiento)
df_sent = pd.concat([df.copy(), sentiment_columns], axis=1)
df_sent["categoria"] = df_sent["target"].map(
    {0: "No desastre", 1: "Desastre real"}
)

print("\nDistribucion general del sentimiento:")
print(df_sent["sentiment_label"].value_counts())
print("\nPorcentaje:")
print((df_sent["sentiment_label"].value_counts(normalize=True) * 100).round(2))

# Guardar dataset con listas convertidas a texto.
df_sent_csv = df_sent.copy()
for col in ["positive_words", "negative_words", "neutral_words"]:
    df_sent_csv[col] = df_sent_csv[col].apply(lambda values: " | ".join(values))

df_sent_csv.to_csv(os.path.join(DATA_DIR, "train_sentiment.csv"), index=False)

# ------------------------------------------------------------


## 12. PALABRAS POSITIVAS, NEGATIVAS Y NEUTRAS


In [ ]:
# ------------------------------------------------------------

contador_pos = Counter(
    word for words in df_sent["positive_words"] for word in words
)
contador_neg = Counter(
    word for words in df_sent["negative_words"] for word in words
)
contador_neu = Counter(
    word for words in df_sent["neutral_words"] for word in words
)

top_positive_words = pd.DataFrame(
    contador_pos.most_common(20), columns=["palabra", "frecuencia"]
)
top_negative_words = pd.DataFrame(
    contador_neg.most_common(20), columns=["palabra", "frecuencia"]
)
top_neutral_words = pd.DataFrame(
    contador_neu.most_common(20), columns=["palabra", "frecuencia"]
)

print("\nTOP 20 PALABRAS POSITIVAS")
print(top_positive_words.to_string(index=False))
print("\nTOP 20 PALABRAS NEGATIVAS")
print(top_negative_words.to_string(index=False))
print("\nTOP 20 PALABRAS NEUTRAS")
print(top_neutral_words.to_string(index=False))

pd.concat(
    [
        top_positive_words.assign(tipo="Positiva"),
        top_negative_words.assign(tipo="Negativa"),
        top_neutral_words.assign(tipo="Neutra"),
    ],
    ignore_index=True,
).to_csv(os.path.join(DATA_DIR, "top_palabras_sentimiento.csv"), index=False)

# Distribucion de sentimiento por categoria.
sentiment_by_category = (
    pd.crosstab(
        df_sent["categoria"],
        df_sent["sentiment_label"],
        normalize="index",
    )
    * 100
)

print("\nSentimiento por categoria (%):")
print(sentiment_by_category.round(2))
sentiment_by_category.round(2).to_csv(
    os.path.join(DATA_DIR, "sentimiento_por_categoria.csv")
)

# Garantizar columnas aunque alguna categoria no exista.
for col in ["Negativo", "Neutro", "Positivo"]:
    if col not in sentiment_by_category.columns:
        sentiment_by_category[col] = 0.0

sentiment_by_category[["Negativo", "Neutro", "Positivo"]].plot.bar(
    figsize=(8, 5)
)
plt.title("Distribucion del sentimiento por categoria")
plt.ylabel("Porcentaje de tweets")
plt.xlabel("Categoria")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "13_sentimiento_por_categoria.png"), dpi=300)
plt.show()

# ------------------------------------------------------------


## 13. TOP 10 TWEETS MAS NEGATIVOS Y POSITIVOS


In [ ]:
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("11. TOP 10 TWEETS MAS NEGATIVOS Y POSITIVOS")
print("=" * 80)

columns_top = [
    "text",
    "target",
    "categoria",
    "sentiment_polarity",
    "negativity",
    "positivity",
    "sentiment_label",
]

top_10_neg = (
    df_sent.sort_values(
        ["sentiment_polarity", "negativity"], ascending=[True, False]
    )
    .head(10)[columns_top]
    .reset_index(drop=True)
)

top_10_pos = (
    df_sent.sort_values(
        ["sentiment_polarity", "positivity"], ascending=[False, False]
    )
    .head(10)[columns_top]
    .reset_index(drop=True)
)

print("\n10 TWEETS MAS NEGATIVOS")
print(top_10_neg.to_string(index=False))
print("\nCategorias:")
print(top_10_neg["categoria"].value_counts())

print("\n10 TWEETS MAS POSITIVOS")
print(top_10_pos.to_string(index=False))
print("\nCategorias:")
print(top_10_pos["categoria"].value_counts())

top_10_neg.to_csv(os.path.join(DATA_DIR, "top_10_tweets_negativos.csv"), index=False)
top_10_pos.to_csv(os.path.join(DATA_DIR, "top_10_tweets_positivos.csv"), index=False)

# ------------------------------------------------------------


## 14. ¿LOS TWEETS DE DESASTRE SON MAS NEGATIVOS?


In [ ]:
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("12. COMPARACION DE NEGATIVIDAD ENTRE CATEGORIAS")
print("=" * 80)

neg_summary = (
    df_sent.groupby("categoria")["negativity"]
    .agg(["count", "mean", "median", "std"])
    .rename(
        columns={
            "count": "n",
            "mean": "media",
            "median": "mediana",
            "std": "desviacion_estandar",
        }
    )
)

print(neg_summary.round(4))
neg_summary.to_csv(os.path.join(DATA_DIR, "resumen_negatividad_categoria.csv"))

neg_disaster = df_sent.loc[df_sent["target"] == 1, "negativity"]
neg_non_disaster = df_sent.loc[df_sent["target"] == 0, "negativity"]

mean_neg_disaster = float(neg_disaster.mean())
mean_neg_non_disaster = float(neg_non_disaster.mean())
neg_difference = mean_neg_disaster - mean_neg_non_disaster

u_stat, p_value = mannwhitneyu(
    neg_disaster,
    neg_non_disaster,
    alternative="greater",
)

print(f"\nNegatividad promedio - Desastre real: {mean_neg_disaster:.4f}")
print(f"Negatividad promedio - No desastre  : {mean_neg_non_disaster:.4f}")
print(f"Diferencia                           : {neg_difference:+.4f}")
print(f"Mann-Whitney U                       : {u_stat:.0f}")
print(f"p-value (H1: desastre > no desastre) : {p_value:.6g}")

if mean_neg_disaster > mean_neg_non_disaster:
    print("\nRespuesta: los tweets de desastre son, en promedio, mas negativos.")
else:
    print("\nRespuesta: los tweets de desastre NO son, en promedio, mas negativos.")

if p_value < 0.05:
    print("La diferencia es estadisticamente significativa al 5%.")
else:
    print("La diferencia no es estadisticamente significativa al 5%.")

# Figuras negatividad.
plt.figure(figsize=(7, 5))
sns.boxplot(data=df_sent, x="categoria", y="negativity")
plt.title("Negatividad de los tweets segun categoria")
plt.xlabel("Categoria")
plt.ylabel("Negatividad (0 a 1)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "14_negatividad_boxplot.png"), dpi=300)
plt.show()

mean_plot = (
    df_sent.groupby("categoria", as_index=False)["negativity"].mean()
)
plt.figure(figsize=(7, 5))
plt.bar(mean_plot["categoria"], mean_plot["negativity"])
plt.title("Negatividad promedio segun categoria")
plt.xlabel("Categoria")
plt.ylabel("Negatividad promedio")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "15_negatividad_promedio.png"), dpi=300)
plt.show()

# ------------------------------------------------------------


## 15. MODELO ORIGINAL VS. MODELO + NEGATIVIDAD


In [ ]:
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("13. MODELO ORIGINAL VS MODELO CON NEGATIVIDAD")
print("=" * 80)

# Agregar la negatividad al train/test usando los indices originales del split.
train_sent = df_sent.iloc[train_idx].copy()
test_sent = df_sent.iloc[test_idx].copy()

y_train_sent = train_sent["target"]
y_test_sent = test_sent["target"]
groups_train_sent = train_sent["text_clean"]

# Extraer el clasificador ya seleccionado y ajustado por CV.
best_classifier = clone(mejor_pipeline.named_steps["clf"])

# Modelo base (mismo algoritmo e hiperparametros elegidos, solo TF-IDF).
base_compare_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(max_features=5000, ngram_range=(1, 2)),
        ),
        ("clf", clone(best_classifier)),
    ]
)

# Modelo con TF-IDF + negativity.
feature_transformer = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(max_features=5000, ngram_range=(1, 2)),
            "text_clean",
        ),
        ("negativity", "passthrough", ["negativity"]),
    ],
    remainder="drop",
)

neg_compare_pipeline = Pipeline(
    [
        ("features", feature_transformer),
        ("clf", clone(best_classifier)),
    ]
)

# Comparar los dos enfoques con CV SOLO en entrenamiento.
base_cv_scores = cross_val_score(
    base_compare_pipeline,
    train_sent["text_clean"],
    y_train_sent,
    groups=groups_train_sent,
    cv=cv,
    scoring="f1_macro",
    n_jobs=1,
)

neg_cv_scores = cross_val_score(
    neg_compare_pipeline,
    train_sent[["text_clean", "negativity"]],
    y_train_sent,
    groups=groups_train_sent,
    cv=cv,
    scoring="f1_macro",
    n_jobs=1,
)

base_cv_mean = float(base_cv_scores.mean())
neg_cv_mean = float(neg_cv_scores.mean())

print(f"F1 macro CV - Modelo original       : {base_cv_mean:.4f}")
print(f"F1 macro CV - Modelo + negatividad  : {neg_cv_mean:.4f}")
print(f"Cambio CV                           : {(neg_cv_mean - base_cv_mean):+.4f}")

# Entrenar ambos modelos sobre TODO el train y evaluar sobre el MISMO test.
base_compare_pipeline.fit(train_sent["text_clean"], y_train_sent)
pred_base_compare = base_compare_pipeline.predict(test_sent["text_clean"])

neg_compare_pipeline.fit(
    train_sent[["text_clean", "negativity"]],
    y_train_sent,
)
pred_neg_compare = neg_compare_pipeline.predict(
    test_sent[["text_clean", "negativity"]]
)


def model_metrics(y_true, y_pred, name):
    return {
        "modelo": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro"),
        "recall_macro": recall_score(y_true, y_pred, average="macro"),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "precision_desastre": precision_score(y_true, y_pred, pos_label=1),
        "recall_desastre": recall_score(y_true, y_pred, pos_label=1),
        "f1_desastre": f1_score(y_true, y_pred, pos_label=1),
    }


comparison_df = pd.DataFrame(
    [
        model_metrics(
            y_test_sent,
            pred_base_compare,
            f"{mejor_nombre} - solo TF-IDF",
        ),
        model_metrics(
            y_test_sent,
            pred_neg_compare,
            f"{mejor_nombre} - TF-IDF + negatividad",
        ),
    ]
).set_index("modelo")

comparison_df["f1_macro_cv_train"] = [base_cv_mean, neg_cv_mean]
comparison_df.to_csv(
    os.path.join(DATA_DIR, "resultados_modelo_con_negatividad.csv")
)

print("\nComparacion en test:")
print(comparison_df.round(4))

f1_base_test = float(comparison_df.iloc[0]["f1_macro"])
f1_neg_test = float(comparison_df.iloc[1]["f1_macro"])
f1_change_pp = (f1_neg_test - f1_base_test) * 100

print(f"\nCambio de F1 macro en test: {f1_change_pp:+.2f} puntos porcentuales")

if neg_cv_mean > base_cv_mean:
    print("Segun CV de entrenamiento, la negatividad SI aporta mejora.")
    use_negativity_final = True
else:
    print("Segun CV de entrenamiento, la negatividad NO mejora el modelo.")
    use_negativity_final = False

# Grafica comparativa.
comparison_df[
    ["accuracy", "precision_macro", "recall_macro", "f1_macro", "recall_desastre"]
].plot.bar(figsize=(11, 6), width=0.8)
plt.title("Modelo original vs modelo con negatividad")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "16_comparacion_modelo_negatividad.png"), dpi=300)
plt.show()

# Matriz de confusion del modelo + negatividad.
cm_neg = confusion_matrix(y_test_sent, pred_neg_compare)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(
    cm_neg,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No desastre", "Desastre"],
    yticklabels=["No desastre", "Desastre"],
)
plt.title("Matriz de confusion - Modelo con negatividad")
plt.xlabel("Predicho")
plt.ylabel("Real")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "17_confusion_modelo_negatividad.png"), dpi=300)
plt.show()

# ------------------------------------------------------------


## 16. MODELO FINAL PARA CLASIFICAR NUEVOS TWEETS


In [ ]:
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("14. GUARDANDO MODELO FINAL")
print("=" * 80)

if use_negativity_final:
    deploy_input_train = train_sent[["text_clean", "negativity"]]
    deploy_input_test = test_sent[["text_clean", "negativity"]]
    deploy_model_template = clone(neg_compare_pipeline)
else:
    deploy_input_train = train_sent["text_clean"]
    deploy_input_test = test_sent["text_clean"]
    deploy_model_template = clone(base_compare_pipeline)

# Si el clasificador de fondo es LinearSVC, calibrar el pipeline completo.
if isinstance(best_classifier, LinearSVC):
    modelo_final = CalibratedClassifierCV(
        deploy_model_template,
        cv=3,
        method="sigmoid",
    )
else:
    modelo_final = deploy_model_template

modelo_final.fit(deploy_input_train, y_train_sent)
pred_final = modelo_final.predict(deploy_input_test)

final_f1 = f1_score(y_test_sent, pred_final, average="macro")
final_acc = accuracy_score(y_test_sent, pred_final)

if hasattr(modelo_final, "predict_proba"):
    final_proba = modelo_final.predict_proba(deploy_input_test)[:, 1]
    final_auc = roc_auc_score(y_test_sent, final_proba)
else:
    final_auc = np.nan

joblib.dump(modelo_final, os.path.join(MODEL_DIR, "modelo_final_completo.joblib"))

model_info = pd.DataFrame(
    [
        {
            "modelo_algoritmo": mejor_nombre,
            "usa_negatividad": use_negativity_final,
            "f1_macro_cv_base": base_cv_mean,
            "f1_macro_cv_negatividad": neg_cv_mean,
            "f1_macro_test_final": final_f1,
            "accuracy_test_final": final_acc,
            "roc_auc_test_final": final_auc,
            "best_params": str(mejores_parametros),
            "max_features": 5000,
            "ngram_range": "(1, 2)",
            "sentiment_method": "TextBlob polarity",
        }
    ]
)
model_info.to_csv(os.path.join(MODEL_DIR, "info_modelo_final.csv"), index=False)

print("Modelo final guardado en data/modelos/modelo_final_completo.joblib")
print("Usa negatividad:", use_negativity_final)
print("F1 macro test final:", round(final_f1, 4))
print("Accuracy test final:", round(final_acc, 4))
if not np.isnan(final_auc):
    print("ROC-AUC test final:", round(final_auc, 4))


def clasificar_tweet(texto: str):
    """Recibe un tweet sin preprocesar y devuelve la clase y confianza."""
    limpio = clean_text(texto)
    sentiment_result = analizar_sentimiento(texto)
    negativity = float(sentiment_result["negativity"])
    polarity = float(sentiment_result["sentiment_polarity"])
    sentiment_label = sentiment_result["sentiment_label"]

    if use_negativity_final:
        entrada = pd.DataFrame(
            {
                "text_clean": [limpio],
                "negativity": [negativity],
            }
        )
    else:
        entrada = [limpio]

    pred = int(modelo_final.predict(entrada)[0])

    if hasattr(modelo_final, "predict_proba"):
        probs = modelo_final.predict_proba(entrada)[0]
        confidence = float(probs[pred])
    else:
        confidence = np.nan

    label = "DESASTRE" if pred == 1 else "NO DESASTRE"

    return {
        "prediccion": pred,
        "clasificacion": label,
        "confianza": confidence,
        "sentimiento": sentiment_label,
        "polaridad": polarity,
        "negatividad": negativity,
        "texto_limpio": limpio,
    }


print("\nPrueba de la funcion final:")
for tweet in ejemplos:
    result = clasificar_tweet(tweet)
    print("\nTweet:", tweet)
    print("Clasificacion:", result["clasificacion"])
    if not np.isnan(result["confianza"]):
        print("Confianza:", round(result["confianza"], 3))
    print("Sentimiento:", result["sentimiento"])
    print("Negatividad:", round(result["negatividad"], 3))

# ------------------------------------------------------------


## 17. PREDICCIONES OPCIONALES PARA test.csv DE KAGGLE


In [ ]:
# ------------------------------------------------------------

if os.path.exists(TEST_KAGGLE_PATH):
    kaggle_test = pd.read_csv(TEST_KAGGLE_PATH)
    kaggle_test["text_clean"] = kaggle_test["text"].apply(clean_text)

    if use_negativity_final:
        kaggle_sent = kaggle_test["text"].apply(analizar_sentimiento)
        kaggle_test["negativity"] = kaggle_sent["negativity"].values
        kaggle_input = kaggle_test[["text_clean", "negativity"]]
    else:
        kaggle_input = kaggle_test["text_clean"]

    kaggle_pred = modelo_final.predict(kaggle_input).astype(int)

    submission = pd.DataFrame(
        {
            "id": kaggle_test["id"],
            "target": kaggle_pred,
        }
    )
    submission.to_csv(os.path.join(DATA_DIR, "predicciones_test_kaggle.csv"), index=False)
    print("\nPredicciones opcionales de test.csv guardadas en data/predicciones_test_kaggle.csv")

# ------------------------------------------------------------


## 18. REQUIREMENTS PARA REPRODUCIBILIDAD


In [ ]:
# ------------------------------------------------------------

packages_for_requirements = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "nltk",
    "wordcloud",
    "textblob",
    "scipy",
    "joblib",
]

requirements_lines = []
for pkg in packages_for_requirements:
    try:
        requirements_lines.append(f"{pkg}=={version(pkg)}")
    except PackageNotFoundError:
        pass

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(requirements_lines) + "\n")

# ------------------------------------------------------------


## 19. RESUMEN FINAL PARA COPIAR AL INFORME


In [ ]:
# ------------------------------------------------------------

negative_top_counts = top_10_neg["categoria"].value_counts()
positive_top_counts = top_10_pos["categoria"].value_counts()

summary = {
    "filas_originales": len(raw_df),
    "filas_modeladas": len(df),
    "train_n": len(train_df),
    "test_n": len(test_df),
    "mejor_modelo_cv": mejor_nombre,
    "mejor_f1_macro_cv": mejor_f1_cv,
    "accuracy_test_modelo_base": base_metrics["accuracy"],
    "f1_macro_test_modelo_base": base_metrics["f1_macro"],
    "roc_auc_test_modelo_base": roc_auc,
    "negatividad_media_desastre": mean_neg_disaster,
    "negatividad_media_no_desastre": mean_neg_non_disaster,
    "diferencia_negatividad": neg_difference,
    "mannwhitney_pvalue": p_value,
    "f1_macro_cv_sin_negatividad": base_cv_mean,
    "f1_macro_cv_con_negatividad": neg_cv_mean,
    "f1_macro_test_sin_negatividad": f1_base_test,
    "f1_macro_test_con_negatividad": f1_neg_test,
    "cambio_f1_test_pp": f1_change_pp,
    "modelo_final_usa_negatividad": use_negativity_final,
    "f1_macro_test_final": final_f1,
    "accuracy_test_final": final_acc,
    "roc_auc_test_final": final_auc,
    "top10_negativos_desastre": int(negative_top_counts.get("Desastre real", 0)),
    "top10_negativos_no_desastre": int(negative_top_counts.get("No desastre", 0)),
    "top10_positivos_desastre": int(positive_top_counts.get("Desastre real", 0)),
    "top10_positivos_no_desastre": int(positive_top_counts.get("No desastre", 0)),
}

pd.DataFrame([summary]).to_csv(
    os.path.join(DATA_DIR, "resumen_final_laboratorio5.csv"), index=False
)

print("\n" + "=" * 80)
print("RESUMEN FINAL PARA EL INFORME")
print("=" * 80)
print(f"Dataset original: {len(raw_df)} tweets")
print(f"Dataset utilizado despues de limpieza: {len(df)} tweets")
print(f"Train/Test: {len(train_df)} / {len(test_df)}")
print(f"Mejor modelo segun CV: {mejor_nombre}")
print(f"Mejor F1 macro CV: {mejor_f1_cv:.4f}")
print(f"F1 macro test modelo base: {base_metrics['f1_macro']:.4f}")
if not np.isnan(roc_auc):
    print(f"ROC-AUC modelo base: {roc_auc:.4f}")
print(f"Negatividad promedio desastre: {mean_neg_disaster:.4f}")
print(f"Negatividad promedio no desastre: {mean_neg_non_disaster:.4f}")
print(f"Diferencia de negatividad: {neg_difference:+.4f}")
print(f"p-value Mann-Whitney: {p_value:.6g}")
print(f"F1 macro CV sin negatividad: {base_cv_mean:.4f}")
print(f"F1 macro CV con negatividad: {neg_cv_mean:.4f}")
print(f"F1 macro test sin negatividad: {f1_base_test:.4f}")
print(f"F1 macro test con negatividad: {f1_neg_test:.4f}")
print(f"Cambio F1 test: {f1_change_pp:+.2f} puntos porcentuales")
print(f"El modelo final usa negatividad: {use_negativity_final}")
print(f"F1 macro test final: {final_f1:.4f}")
print(f"Accuracy test final: {final_acc:.4f}")
if not np.isnan(final_auc):
    print(f"ROC-AUC test final: {final_auc:.4f}")
print("\nTop 10 negativos por categoria:")
print(negative_top_counts)
print("\nTop 10 positivos por categoria:")
print(positive_top_counts)

print("\nArchivos importantes generados:")
for path in [
    "data/train_clean.csv",
    "data/train_sentiment.csv",
    "data/resultados_cv_modelos.csv",
    "data/resultados_gridsearch.csv",
    "data/top_10_tweets_negativos.csv",
    "data/top_10_tweets_positivos.csv",
    "data/resumen_negatividad_categoria.csv",
    "data/resultados_modelo_con_negatividad.csv",
    "data/resumen_final_laboratorio5.csv",
    "data/modelos/modelo_final_completo.joblib",
    "data/modelos/info_modelo_final.csv",
    "requirements.txt",
]:
    print(" -", path)

print("\nLISTO: el codigo cubre los ejercicios 1 al 10 del laboratorio y deja los resultados listos para el informe final.")
